# LeetCode 697 — Degree of an Array bằng Python

**Link đề:** https://leetcode.com/problems/degree-of-an-array/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table

```python
class Solution:
    def findShortestSubArray(self, nums: List[int]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **degree** (bậc) của mảng = tần suất lớn nhất của một phần tử.
2. Dùng `dict` / `Counter` để đếm tần suất, và `dict` để nhớ **vị trí xuất hiện đầu tiên / cuối cùng**.
3. Thấy vì sao subarray ngắn nhất cùng degree **phải** đi từ lần xuất hiện đầu đến lần cuối của một phần tử có tần suất max.
4. Cài đặt 4 cách: brute force mọi subarray → `Counter` + `first/last` (2 lượt) → một lượt duyệt → gói gọn bằng `index`.
5. Trace, chứng minh đúng, Time/Space, các lỗi hay gặp, bài tập biến thể.

## 1. Đề bài và ví dụ

Cho mảng `nums` khác rỗng gồm số nguyên không âm. **Degree** của mảng là tần suất lớn nhất của một phần tử bất kỳ.

Tìm **độ dài nhỏ nhất** của một subarray **liên tiếp** của `nums` có cùng degree với `nums`.

| `nums` | degree | Kết quả | Giải thích |
|---|:---:|:---:|---|
| `[1,2,2,3,1]` | 2 | `2` | `1` và `2` đều xuất hiện 2 lần; `[2,2]` dài 2 là ngắn nhất |
| `[1,2,2,3,1,4,2]` | 3 | `6` | `2` xuất hiện 3 lần; `[2,2,3,1,4,2]` dài 6 |

**Constraints:** `1 <= len(nums) <= 50000`, `0 <= nums[i] <= 49999`.

In [1]:
nums1 = [1, 2, 2, 3, 1]
nums2 = [1, 2, 2, 3, 1, 4, 2]
print("nums1:", nums1, "| độ dài:", len(nums1))
print("nums2:", nums2, "| độ dài:", len(nums2))
print("số lần 2 xuất hiện trong nums2:", nums2.count(2))

nums1: [1, 2, 2, 3, 1] | độ dài: 5
nums2: [1, 2, 2, 3, 1, 4, 2] | độ dài: 7
số lần 2 xuất hiện trong nums2: 3


## 2. Nền tảng Python

### 2.1. Subarray và slicing list

Subarray = đoạn **liên tiếp** `nums[i:j]` (từ `i` đến trước `j`), độ dài `j - i`. Khác với subsequence (được bỏ phần tử ở giữa).

In [2]:
nums = [1, 2, 2, 3, 1, 4, 2]
print(nums[1:7])        # [2, 2, 3, 1, 4, 2]
print(len(nums[1:7]))   # 6
print(nums[2:4])        # [2, 3]

[2, 2, 3, 1, 4, 2]
6
[2, 3]


### 2.2. `dict` đếm tần suất bằng tay

Ý tưởng: duyệt từng số `x`, tăng `count[x]`. Dùng `count.get(x, 0)` để lấy giá trị cũ, trả `0` nếu chưa có khóa.

In [3]:
count = {}
for x in [1, 2, 2, 3, 1]:
    count[x] = count.get(x, 0) + 1
print(count)                  # {1: 2, 2: 2, 3: 1}
print(max(count.values()))    # degree = 2

{1: 2, 2: 2, 3: 1}
2


### 2.3. `collections.Counter`

`Counter(nums)` là một dict đặc biệt làm sẵn việc đếm. `.most_common(k)` trả `k` phần tử nhiều nhất.

In [4]:
from collections import Counter
c = Counter([1, 2, 2, 3, 1, 4, 2])
print(c)
print("c[2] =", c[2], "| c[99] =", c[99], "(khóa chưa có -> 0, không lỗi)")
print("most_common(1):", c.most_common(1))

Counter({2: 3, 1: 2, 3: 1, 4: 1})
c[2] = 3 | c[99] = 0 (khóa chưa có -> 0, không lỗi)
most_common(1): [(2, 3)]


### 2.4. `enumerate` — lấy cả chỉ số lẫn giá trị

`for i, x in enumerate(nums)` cho `i = 0, 1, 2, ...` và `x = nums[i]`.

### 2.5. Ghi nhớ vị trí đầu tiên: `setdefault`

`first.setdefault(x, i)`: nếu `x` **chưa** có trong `first` thì gán `first[x] = i`; nếu có rồi thì **giữ nguyên**. Đúng thứ ta cần cho "lần xuất hiện đầu tiên". Còn "lần cuối" thì chỉ cần gán đè `last[x] = i` mỗi lần.

In [5]:
first, last = {}, {}
for i, x in enumerate([1, 2, 2, 3, 1, 4, 2]):
    first.setdefault(x, i)
    last[x] = i
print("first:", first)
print("last :", last)

first: {1: 0, 2: 1, 3: 3, 4: 5}
last : {1: 4, 2: 6, 3: 3, 4: 5}


### 2.6. `list.index(x)` và tìm từ cuối

- `nums.index(x)` trả chỉ số **đầu tiên** của `x` (duyệt `O(n)`).
- Chỉ số cuối: `len(nums) - 1 - nums[::-1].index(x)` (đảo list rồi tìm).

In [6]:
nums = [1, 2, 2, 3, 1, 4, 2]
print("đầu tiên của 2:", nums.index(2))
print("cuối cùng của 2:", len(nums) - 1 - nums[::-1].index(2))

đầu tiên của 2: 1
cuối cùng của 2: 6


## 3. Cách 1 — Brute force: thử mọi subarray

Ý tưởng trực giác:
1. Tính `degree` của cả mảng.
2. Duyệt mọi `(i, j)`, tính degree của `nums[i:j+1]`; nếu bằng `degree` thì cập nhật độ dài nhỏ nhất.

Có thể tăng dần `j` và đếm cộng dồn để mỗi subarray tốn `O(1)` cập nhật → tổng `O(n²)`. Với `n = 50000` là khoảng `1.25·10^9` bước — quá chậm.

In [7]:
def shortest_brute(nums: list[int]) -> int:
    degree = max(Counter(nums).values())
    n = len(nums)
    best = n
    for i in range(n):
        cnt = {}
        cur_max = 0
        for j in range(i, n):
            cnt[nums[j]] = cnt.get(nums[j], 0) + 1
            cur_max = max(cur_max, cnt[nums[j]])
            if cur_max == degree:
                best = min(best, j - i + 1)
                break          # kéo dài thêm j chỉ làm subarray dài hơn
    return best

print(shortest_brute([1, 2, 2, 3, 1]))         # 2
print(shortest_brute([1, 2, 2, 3, 1, 4, 2]))   # 6

2
6


Chú ý `break`: với `i` cố định, `j` đầu tiên làm `cur_max == degree` cho subarray ngắn nhất bắt đầu tại `i`; đi tiếp chỉ dài hơn nên dừng sớm.

## 4. Nút thắt và hint của đề

Hint: *"Giả sử 5 là phần tử duy nhất xuất hiện nhiều nhất, ví dụ `[1, 5, 2, 3, 5, 4, 5, 6]`. Đáp án là gì?"*

Subarray muốn có degree 3 thì phải chứa **cả 3 số 5** (vì chỉ số 5 mới đạt tần suất 3). Đoạn ngắn nhất chứa cả 3 số 5 đi từ số 5 đầu (chỉ số 1) đến số 5 cuối (chỉ số 6) → độ dài `6 - 1 + 1 = 6`.

In [8]:
nums = [1, 5, 2, 3, 5, 4, 5, 6]
first5 = nums.index(5)
last5 = len(nums) - 1 - nums[::-1].index(5)
print("first =", first5, "| last =", last5, "| độ dài =", last5 - first5 + 1)
print("brute:", shortest_brute(nums))

first = 1 | last = 6 | độ dài = 6
brute: 6


## 5. Insight

1. Subarray có degree `d` thì phải có một phần tử `x` xuất hiện `d` lần trong nó. Vì `x` chỉ xuất hiện tối đa `d` lần trong cả mảng → subarray chứa **mọi** lần xuất hiện của `x` → độ dài ≥ `last[x] - first[x] + 1`.
2. Ngược lại, đoạn `nums[first[x] : last[x]+1]` chứa đủ `d` lần `x`, và không phần tử nào vượt `d` (vì `d` là max của cả mảng) → degree đúng bằng `d`.

**Kết luận:**
```text
đáp án = min( last[x] - first[x] + 1 )   với mọi x có count[x] == degree
```

Ta cần 3 bảng: `count`, `first`, `last`.

## 6. Cách 2 — Hai lượt: đếm + first/last, rồi lấy min (lời giải chính)

Mổ xẻ từng đoạn:
- **Lượt 1:** duyệt `enumerate(nums)`, cập nhật `count[x] += 1`, `first.setdefault(x, i)`, `last[x] = i`.
- **Tính degree:** `degree = max(count.values())`.
- **Lượt 2:** duyệt các khóa `x` của `count`; nếu `count[x] == degree` thì xét `last[x] - first[x] + 1`.

In [9]:
def shortest_two_pass(nums: list[int]) -> int:
    count, first, last = {}, {}, {}
    for i, x in enumerate(nums):
        count[x] = count.get(x, 0) + 1
        first.setdefault(x, i)
        last[x] = i
    degree = max(count.values())
    best = len(nums)
    for x, c in count.items():
        if c == degree:
            best = min(best, last[x] - first[x] + 1)
    return best

print(shortest_two_pass([1, 2, 2, 3, 1]))         # 2
print(shortest_two_pass([1, 2, 2, 3, 1, 4, 2]))   # 6

2
6


`count.items()` trả các cặp `(khóa, giá trị)`, nên `for x, c in count.items()` lấy được cả số `x` và tần suất `c`.

## 7. TRACE từng bước — `nums = [1, 2, 2, 3, 1, 4, 2]`

**Lượt 1** (đọc từ trái sang phải):

| i | x | count sau | first sau | last sau |
|---|---|---|---|---|
| 0 | 1 | {1:1} | {1:0} | {1:0} |
| 1 | 2 | {1:1, 2:1} | {1:0, 2:1} | {1:0, 2:1} |
| 2 | 2 | {1:1, 2:2} | không đổi | {1:0, 2:2} |
| 3 | 3 | {.., 3:1} | {.., 3:3} | {.., 3:3} |
| 4 | 1 | {1:2, ..} | không đổi | {1:4, ..} |
| 5 | 4 | {.., 4:1} | {.., 4:5} | {.., 4:5} |
| 6 | 2 | {2:3, ..} | không đổi | {2:6, ..} |

degree = 3. **Lượt 2:** chỉ `x = 2` có count 3 → `6 - 1 + 1 = 6`.

In [10]:
def trace_two_pass(nums: list[int]) -> int:
    count, first, last = {}, {}, {}
    for i, x in enumerate(nums):
        count[x] = count.get(x, 0) + 1
        first.setdefault(x, i)
        last[x] = i
        print(f"i={i}, x={x} | count={count} | first={first} | last={last}")
    degree = max(count.values())
    print("degree =", degree)
    best = len(nums)
    for x, c in count.items():
        if c == degree:
            length = last[x] - first[x] + 1
            print(f"  ứng viên x={x}: first={first[x]}, last={last[x]}, độ dài={length}")
            best = min(best, length)
    print("đáp án =", best)
    return best

trace_two_pass([1, 2, 2, 3, 1, 4, 2])

i=0, x=1 | count={1: 1} | first={1: 0} | last={1: 0}
i=1, x=2 | count={1: 1, 2: 1} | first={1: 0, 2: 1} | last={1: 0, 2: 1}
i=2, x=2 | count={1: 1, 2: 2} | first={1: 0, 2: 1} | last={1: 0, 2: 2}
i=3, x=3 | count={1: 1, 2: 2, 3: 1} | first={1: 0, 2: 1, 3: 3} | last={1: 0, 2: 2, 3: 3}
i=4, x=1 | count={1: 2, 2: 2, 3: 1} | first={1: 0, 2: 1, 3: 3} | last={1: 4, 2: 2, 3: 3}
i=5, x=4 | count={1: 2, 2: 2, 3: 1, 4: 1} | first={1: 0, 2: 1, 3: 3, 4: 5} | last={1: 4, 2: 2, 3: 3, 4: 5}
i=6, x=2 | count={1: 2, 2: 3, 3: 1, 4: 1} | first={1: 0, 2: 1, 3: 3, 4: 5} | last={1: 4, 2: 6, 3: 3, 4: 5}
degree = 3
  ứng viên x=2: first=1, last=6, độ dài=6
đáp án = 6


6

Ví dụ có **nhiều** phần tử cùng đạt degree: `[1, 2, 2, 3, 1]` — cả `1` (đoạn dài 5) và `2` (đoạn dài 2) đều là ứng viên, ta chọn min = 2.

In [11]:
trace_two_pass([1, 2, 2, 3, 1])

i=0, x=1 | count={1: 1} | first={1: 0} | last={1: 0}
i=1, x=2 | count={1: 1, 2: 1} | first={1: 0, 2: 1} | last={1: 0, 2: 1}
i=2, x=2 | count={1: 1, 2: 2} | first={1: 0, 2: 1} | last={1: 0, 2: 2}
i=3, x=3 | count={1: 1, 2: 2, 3: 1} | first={1: 0, 2: 1, 3: 3} | last={1: 0, 2: 2, 3: 3}
i=4, x=1 | count={1: 2, 2: 2, 3: 1} | first={1: 0, 2: 1, 3: 3} | last={1: 4, 2: 2, 3: 3}
degree = 2
  ứng viên x=1: first=0, last=4, độ dài=5
  ứng viên x=2: first=1, last=2, độ dài=2
đáp án = 2


2

## 8. Cách 3 — Một lượt duyệt (cập nhật đáp án ngay khi đếm)

Có thể gộp hai lượt: khi đếm tới `x`, nếu `count[x]` **vượt** degree hiện tại → degree mới, đáp án reset thành `i - first[x] + 1`; nếu **bằng** degree hiện tại → so min.

Tại sao đúng? Tại thời điểm `i`, `i - first[x] + 1` là độ dài đoạn từ lần đầu đến lần **hiện tại** của `x`. Khi `count[x]` đạt giá trị cuối cùng của nó, `i` chính là `last[x]`.

In [12]:
def shortest_one_pass(nums: list[int]) -> int:
    count, first = {}, {}
    degree = best = 0
    for i, x in enumerate(nums):
        first.setdefault(x, i)
        count[x] = count.get(x, 0) + 1
        length = i - first[x] + 1
        if count[x] > degree:
            degree, best = count[x], length
        elif count[x] == degree:
            best = min(best, length)
    return best

print(shortest_one_pass([1, 2, 2, 3, 1]))
print(shortest_one_pass([1, 2, 2, 3, 1, 4, 2]))

2
6


In [13]:
def trace_one_pass(nums: list[int]) -> int:
    count, first = {}, {}
    degree = best = 0
    for i, x in enumerate(nums):
        first.setdefault(x, i)
        count[x] = count.get(x, 0) + 1
        length = i - first[x] + 1
        if count[x] > degree:
            degree, best = count[x], length
            action = "degree mới"
        elif count[x] == degree:
            best = min(best, length)
            action = "bằng degree -> so min"
        else:
            action = "bỏ qua"
        print(f"i={i}, x={x}, count[x]={count[x]}, đoạn={length} | {action:22} | degree={degree}, best={best}")
    return best

trace_one_pass([1, 2, 2, 3, 1])

i=0, x=1, count[x]=1, đoạn=1 | degree mới             | degree=1, best=1
i=1, x=2, count[x]=1, đoạn=1 | bằng degree -> so min  | degree=1, best=1
i=2, x=2, count[x]=2, đoạn=2 | degree mới             | degree=2, best=2
i=3, x=3, count[x]=1, đoạn=1 | bỏ qua                 | degree=2, best=2
i=4, x=1, count[x]=2, đoạn=5 | bằng degree -> so min  | degree=2, best=2


2

## 9. Cách 4 — Ngắn gọn bằng `Counter` + `index` (dễ viết, chậm hơn)

Chỉ dùng công cụ có sẵn: tìm các phần tử đạt degree, với mỗi phần tử dùng `index` từ đầu và từ cuối. Mỗi lần `index` là `O(n)`, có thể có tới `k` ứng viên → `O(k·n)`, trường hợp xấu (mọi phần tử khác nhau, degree 1) là `O(n²)`.

In [14]:
def shortest_index(nums: list[int]) -> int:
    c = Counter(nums)
    degree = max(c.values())
    rev = nums[::-1]
    n = len(nums)
    return min((n - 1 - rev.index(x)) - nums.index(x) + 1
               for x, v in c.items() if v == degree)

print(shortest_index([1, 2, 2, 3, 1]), shortest_index([1, 2, 2, 3, 1, 4, 2]))

2 6


## 10. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def findShortestSubArray(self, nums: List[int]) -> int:
        count, first, last = {}, {}, {}
        for i, x in enumerate(nums):
            count[x] = count.get(x, 0) + 1
            first.setdefault(x, i)
            last[x] = i
        degree = max(count.values())
        return min(last[x] - first[x] + 1 for x in count if count[x] == degree)

sol = Solution()
print(sol.findShortestSubArray([1, 2, 2, 3, 1]))        # 2
print(sol.findShortestSubArray([1, 2, 2, 3, 1, 4, 2]))  # 6

2
6


`from typing import List`: LeetCode tự import sẵn `List`; trong notebook ta import để chú thích kiểu `List[int]` chạy được. `min(... for x in count if ...)` là **generator expression**: giống list comprehension nhưng không tạo list trung gian.

Bản một lượt cũng nộp được:

```python
class Solution:
    def findShortestSubArray(self, nums: List[int]) -> int:
        count, first = {}, {}
        degree = best = 0
        for i, x in enumerate(nums):
            first.setdefault(x, i)
            count[x] = count.get(x, 0) + 1
            length = i - first[x] + 1
            if count[x] > degree:
                degree, best = count[x], length
            elif count[x] == degree:
                best = min(best, length)
        return best
```

## 11. Vì sao đúng?

Gọi `d` = degree của `nums`, `S` = tập các `x` có `count[x] = d`.

- **Cận dưới:** Subarray `T` có degree `d` ⇒ tồn tại `x` xuất hiện `d` lần trong `T`. Vì `count[x] ≤ d` trong cả mảng, nên `count[x] = d` (tức `x ∈ S`) và `T` chứa **tất cả** lần xuất hiện của `x` ⇒ `T` phủ `[first[x], last[x]]` ⇒ `len(T) ≥ last[x] - first[x] + 1`.
- **Đạt được:** Với mọi `x ∈ S`, đoạn `[first[x], last[x]]` chứa `d` lần `x`; mọi phần tử khác có tần suất ≤ `d` trong cả mảng nên trong đoạn cũng ≤ `d` ⇒ degree đoạn đúng bằng `d`.

Vậy đáp án = `min` của `last[x] - first[x] + 1` trên `x ∈ S`.

**Invariant cách một lượt:** sau khi xử lý `nums[0..i]`, `degree` là degree của tiền tố, `best` là min độ dài đoạn `[first[x], vị trí hiện tại cuối của x]` trên các `x` đạt degree của tiền tố. Khi `i = n-1`, tiền tố là cả mảng.

## 12. Độ phức tạp và so sánh

Gọi `n = len(nums)`, `m` = số giá trị khác nhau (`m ≤ n`).

| Cách | Time | Space | Nhận xét |
|---|---|---|---|
| 1. Brute force (có `break`) | `O(n²)` | `O(m)` | Chỉ để hiểu đề |
| 2. Hai lượt `count/first/last` | `O(n)` trung bình | `O(m)` | **Nên nộp**, dễ chứng minh |
| 3. Một lượt | `O(n)` trung bình | `O(m)` | Gọn, nhưng khó giải thích hơn |
| 4. `Counter` + `index` | `O(k·n)`, xấu `O(n²)` | `O(n)` (list đảo) | Viết nhanh, có thể chậm |

"Trung bình" vì thao tác dict là `O(1)` trung bình (hash table). Vì `0 <= nums[i] <= 49999`, có thể thay dict bằng list độ dài 50000 để được `O(1)` chắc chắn.

## 13. Bộ test — mẫu, biên, phản ví dụ, đối chiếu

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:32} PASS ({actual})")

sol = Solution()
check("mẫu 1", sol.findShortestSubArray([1, 2, 2, 3, 1]), 2)
check("mẫu 2", sol.findShortestSubArray([1, 2, 2, 3, 1, 4, 2]), 6)
check("một phần tử", sol.findShortestSubArray([7]), 1)
check("tất cả khác nhau (degree 1)", sol.findShortestSubArray([3, 1, 4, 5]), 1)
check("tất cả giống nhau", sol.findShortestSubArray([2, 2, 2]), 3)
check("hint đề", sol.findShortestSubArray([1, 5, 2, 3, 5, 4, 5, 6]), 6)
check("2 ứng viên, chọn ngắn", sol.findShortestSubArray([1, 2, 1, 3, 3]), 2)
check("số 0", sol.findShortestSubArray([0, 1, 0]), 3)

mẫu 1                            PASS (2)
mẫu 2                            PASS (6)
một phần tử                      PASS (1)
tất cả khác nhau (degree 1)      PASS (1)
tất cả giống nhau                PASS (3)
hint đề                          PASS (6)
2 ứng viên, chọn ngắn            PASS (2)
số 0                             PASS (3)


In [17]:
import random
rng = random.Random(697)
funcs = [shortest_two_pass, shortest_one_pass, shortest_index, sol.findShortestSubArray]
for _ in range(500):
    arr = [rng.randint(0, 5) for _ in range(rng.randint(1, 15))]
    expected = shortest_brute(arr)
    for f in funcs:
        assert f(arr) == expected, (f.__name__, arr)
print("500 mảng ngẫu nhiên: 4 cách đều khớp brute force")

500 mảng ngẫu nhiên: 4 cách đều khớp brute force


In [18]:
import time
big = [rng.randint(0, 49999) for _ in range(50000)]
t0 = time.perf_counter()
r = sol.findShortestSubArray(big)
print("n = 50000 ->", r, "| thời gian:", round(time.perf_counter() - t0, 4), "giây")

n = 50000 -> 41818 | thời gian: 0.0283 giây


## 14. Lỗi người mới hay gặp

### Lỗi 1 — Trả về degree thay vì độ dài
Đề hỏi **độ dài subarray**, không phải degree. `[1,2,2,3,1]` có degree 2 và đáp án cũng là 2 — trùng hợp dễ làm tưởng đúng! Test thêm `[1,2,2,3,1,4,2]` (degree 3, đáp án 6) sẽ lộ lỗi.

### Lỗi 2 — Chỉ xét **một** phần tử có tần suất max

In [19]:
def wrong_one_candidate(nums):
    c = Counter(nums)
    x = c.most_common(1)[0][0]           # chỉ lấy 1 ứng viên
    first = nums.index(x)
    last = len(nums) - 1 - nums[::-1].index(x)
    return last - first + 1

print("sai:", wrong_one_candidate([1, 2, 2, 3, 1]), "| đúng:", shortest_two_pass([1, 2, 2, 3, 1]))

sai: 5 | đúng: 2


Vì sao sai: `most_common(1)` trả `1` (xuất hiện trước), đoạn dài 5; nhưng `2` cũng đạt degree với đoạn dài 2. Sửa: duyệt **mọi** `x` có `count[x] == degree`.

### Lỗi 3 — Ghi đè `first[x]` mỗi lần gặp

In [20]:
def wrong_first_overwrite(nums):
    count, first, last = {}, {}, {}
    for i, x in enumerate(nums):
        count[x] = count.get(x, 0) + 1
        first[x] = i               # SAI: phải chỉ gán lần đầu
        last[x] = i
    d = max(count.values())
    return min(last[x] - first[x] + 1 for x in count if count[x] == d)

print("sai:", wrong_first_overwrite([1, 2, 2, 3, 1, 4, 2]), "| đúng:", 6)

sai: 1 | đúng: 6


Vì sao sai: `first[x]` luôn bằng `last[x]` → mọi đoạn dài 1. Sửa: `first.setdefault(x, i)` hoặc `if x not in first: first[x] = i`.

### Lỗi 4 — Quên `+ 1` khi tính độ dài
Độ dài đoạn từ `a` đến `b` (bao gồm hai đầu) là `b - a + 1`, không phải `b - a`. Ví dụ `[7]`: `0 - 0 = 0` sai, phải là `1`.

### Lỗi 5 — Cách một lượt: dùng `>=` gộp hai nhánh
Nếu viết `if count[x] >= degree: degree, best = count[x], length` thì khi bằng degree sẽ **ghi đè** `best` bằng đoạn mới (có thể dài hơn) thay vì lấy min.

In [21]:
def wrong_ge(nums):
    count, first = {}, {}
    degree = best = 0
    for i, x in enumerate(nums):
        first.setdefault(x, i)
        count[x] = count.get(x, 0) + 1
        if count[x] >= degree:
            degree, best = count[x], i - first[x] + 1
    return best

print("sai:", wrong_ge([2, 2, 1, 3, 1]), "| đúng:", shortest_two_pass([2, 2, 1, 3, 1]))

sai: 3 | đúng: 2


## 15. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu:** đề nhắc tới "tần suất", "xuất hiện nhiều nhất", "đoạn chứa tất cả lần xuất hiện" → nghĩ tới **dict đếm + dict vị trí đầu/cuối**.

- Bài liên quan: 1 Two Sum (dict vị trí), 169 Majority Element (đếm), 763 Partition Labels (dùng `last[x]`), 2287/2404 (đếm tần suất).
- Mẹo: khi cần "đoạn bao trọn mọi lần xuất hiện của x", chỉ cần `first[x]` và `last[x]`.

### 15.1. Biến thể A — trả về chính subarray (không chỉ độ dài)

In [22]:
def shortest_subarray_itself(nums: list[int]) -> list[int]:
    count, first, last = {}, {}, {}
    for i, x in enumerate(nums):
        count[x] = count.get(x, 0) + 1
        first.setdefault(x, i)
        last[x] = i
    d = max(count.values())
    x_best = min((x for x in count if count[x] == d), key=lambda x: last[x] - first[x])
    return nums[first[x_best]: last[x_best] + 1]

print(shortest_subarray_itself([1, 2, 2, 3, 1, 4, 2]))
print(shortest_subarray_itself([1, 2, 2, 3, 1]))

[2, 2, 3, 1, 4, 2]
[2, 2]


`min(..., key=lambda x: ...)`: chọn phần tử có **giá trị khóa** nhỏ nhất. `lambda x: last[x] - first[x]` là hàm vô danh một dòng.

### 15.2. Biến thể B — dùng list thay dict (vì giá trị bị chặn 0..49999)

In [23]:
def shortest_array_tables(nums: list[int]) -> int:
    M = max(nums) + 1
    count = [0] * M
    first = [-1] * M
    last = [-1] * M
    for i, x in enumerate(nums):
        count[x] += 1
        if first[x] == -1:
            first[x] = i
        last[x] = i
    d = max(count)
    return min(last[x] - first[x] + 1 for x in range(M) if count[x] == d)

print(shortest_array_tables([1, 2, 2, 3, 1, 4, 2]))

6


## 16. Bài tập tự luyện

**Bài 1 (tính tay).** `nums = [4, 1, 4, 2, 2, 3, 2, 4]`. Tính degree và đáp án. Gán vào `ex1_degree`, `ex1_answer`.

**Bài 2 (TODO).** `ex2_degree(nums)`: tính degree không dùng `Counter`.

**Bài 3 (TODO).** `ex3_first_last(nums)`: trả về 2 dict `(first, last)`.

**Bài 4 (tự viết).** `ex4_shortest(nums)`: viết lại lời giải hai lượt.

**Bài 5 (biến thể).** `ex5_count_shortest(nums)`: có bao nhiêu phần tử `x` đạt degree **và** cho đoạn ngắn nhất (đếm số ứng viên hòa nhau).

**Bài 6 (mini challenge).** `ex6_longest_with_degree(nums)`: độ dài subarray **dài nhất** có degree **nhỏ hơn** degree của cả mảng (có thể bằng 0 nếu không có). Gợi ý: cửa sổ trượt.

In [24]:
ex1_degree = None    # TODO
ex1_answer = None    # TODO

def ex2_degree(nums: list[int]) -> int:
    # TODO
    pass

def ex3_first_last(nums: list[int]):
    # TODO: trả (first, last)
    pass

# assert ex2_degree([1, 2, 2, 3, 1, 4, 2]) == 3
# assert ex3_first_last([1, 2, 1]) == ({1: 0, 2: 1}, {1: 2, 2: 1})

In [25]:
def ex4_shortest(nums: list[int]) -> int:
    # TODO
    pass

def ex5_count_shortest(nums: list[int]) -> int:
    # TODO
    pass

def ex6_longest_with_degree(nums: list[int]) -> int:
    # TODO
    pass

# assert ex4_shortest([1, 2, 2, 3, 1, 4, 2]) == 6
# assert ex5_count_shortest([1, 1, 2, 2, 3, 3]) == 3
# assert ex6_longest_with_degree([1, 2, 2, 3, 1]) == 3

## 17. Gợi ý 3 mức và đáp án

### Gợi ý
- **Mức 1:** Mọi bài quy về 3 bảng `count`, `first`, `last`. Bài 6: degree của cửa sổ phải `< d`.
- **Mức 2:** Bài 5 — tính `best` trước, rồi đếm các `x` có `count[x] == d` và độ dài `== best`. Bài 6 — cửa sổ `[l, r]`, khi thêm `nums[r]` làm tần suất của nó đạt `d` thì dịch `l` tới khi tần suất đó `< d`.
- **Mức 3:** Bài 6:
  ```text
  for r, x in enumerate(nums):
      cnt[x] += 1
      while cnt[x] >= d: cnt[nums[l]] -= 1; l += 1
      best = max(best, r - l + 1)
  ```

### Đáp án tham khảo
Bài 1: `4` xuất hiện 3 lần (chỉ số 0, 2, 7), `2` xuất hiện 3 lần (chỉ số 3, 4, 6). Degree = 3. Đoạn của `4` dài 8, của `2` dài `6 - 3 + 1 = 4` → đáp án `4`.

In [26]:
ans_ex1_degree, ans_ex1_answer = 3, 4

def ans_ex2_degree(nums):
    cnt = {}
    for x in nums:
        cnt[x] = cnt.get(x, 0) + 1
    return max(cnt.values())

def ans_ex3_first_last(nums):
    first, last = {}, {}
    for i, x in enumerate(nums):
        if x not in first:
            first[x] = i
        last[x] = i
    return first, last

def ans_ex4_shortest(nums):
    first, last = ans_ex3_first_last(nums)
    cnt = Counter(nums)
    d = max(cnt.values())
    return min(last[x] - first[x] + 1 for x in cnt if cnt[x] == d)

def ans_ex5_count_shortest(nums):
    first, last = ans_ex3_first_last(nums)
    cnt = Counter(nums)
    d = max(cnt.values())
    lengths = [last[x] - first[x] + 1 for x in cnt if cnt[x] == d]
    return lengths.count(min(lengths))

def ans_ex6_longest_with_degree(nums):
    d = ans_ex2_degree(nums)
    cnt = {}
    l = best = 0
    for r, x in enumerate(nums):
        cnt[x] = cnt.get(x, 0) + 1
        while cnt[x] >= d:
            cnt[nums[l]] -= 1
            l += 1
        best = max(best, r - l + 1)
    return best

assert (ans_ex1_degree, ans_ex1_answer) == (3, shortest_brute([4, 1, 4, 2, 2, 3, 2, 4]))
assert ans_ex2_degree([1, 2, 2, 3, 1, 4, 2]) == 3
assert ans_ex3_first_last([1, 2, 1]) == ({1: 0, 2: 1}, {1: 2, 2: 1})
assert ans_ex4_shortest([1, 2, 2, 3, 1, 4, 2]) == 6
assert ans_ex5_count_shortest([1, 1, 2, 2, 3, 3]) == 3
assert ans_ex6_longest_with_degree([1, 2, 2, 3, 1]) == 3
assert ans_ex6_longest_with_degree([5]) == 0

def brute_ex6(nums):
    d = ans_ex2_degree(nums); best = 0
    for i in range(len(nums)):
        for j in range(i, len(nums)):
            if max(Counter(nums[i:j + 1]).values()) < d:
                best = max(best, j - i + 1)
    return best

for _ in range(300):
    arr = [rng.randint(0, 4) for _ in range(rng.randint(1, 10))]
    assert ans_ex4_shortest(arr) == shortest_brute(arr)
    assert ans_ex6_longest_with_degree(arr) == brute_ex6(arr)
print("Đáp án tham khảo: PASS toàn bộ")

Đáp án tham khảo: PASS toàn bộ


## 18. Template nhớ nhanh và checklist

```python
class Solution:
    def findShortestSubArray(self, nums: List[int]) -> int:
        count, first, last = {}, {}, {}
        for i, x in enumerate(nums):
            count[x] = count.get(x, 0) + 1
            first.setdefault(x, i)
            last[x] = i
        degree = max(count.values())
        return min(last[x] - first[x] + 1 for x in count if count[x] == degree)
```

Câu thần chú: **"Đếm – nhớ đầu – nhớ cuối – lấy min đoạn của các phần tử đạt degree."**

### Checklist trước khi submit
- [ ] Trả **độ dài**, không trả degree.
- [ ] Xét **mọi** phần tử đạt degree, không chỉ một.
- [ ] `first[x]` chỉ gán lần đầu (`setdefault`), `last[x]` gán đè.
- [ ] Độ dài = `last - first + 1`.
- [ ] Test `[7]`, mảng toàn khác nhau, mảng toàn giống nhau, nhiều ứng viên hòa.
- [ ] Time `O(n)` trung bình, Space `O(m)`.